In [16]:
from DataBalancing import balancing
from models import get_efficientnet_model, get_resnet_model, get_densenet_model
import pickle
from data_partial_inclusion import get_partial_inclusion_data

import random
import numpy as np
import tensorflow as tf
import os

from tensorflow.keras.optimizers import Adam

In [ ]:
import numpy as np
import tensorflow as tf
import os
import pathlib
import pandas as pd
from DataBalancing import balancing
import random
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.utils import to_categorical
from keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from keras import regularizers
from keras.models import load_model
from sklearn import metrics
import pickle
from sklearn.metrics import confusion_matrix, precision_score
from models import get_efficientnet_model, get_resnet_model, get_densenet_model

In [2]:
## define data root
train_root = "C:/Users/kheir/Downloads/Shortcut_Learning/Train"
## define our data centers
filtered_data_dic = {"Prince Charles Hospital":100, "Roswell Park": 100, "Asterand": 100, "Johns Hopkins": 100, "University of Pittsburgh": 100}
# ex_train_filtered_data_dic = {"Asterand": 100}

cancers = ["Lung Squamous Cell Carcinoma", "Lung Adenocarcinoma"]

In [3]:
#load
filename = './shuffled_reshaped_balanced_images.pickle'
with open(filename, 'rb') as file:
    shuffled_reshaped_balanced_images = pickle.load(file)
filename = './shuffled_balanced_center_labels.pickle'
with open(filename, 'rb') as file:
    shuffled_balanced_center_labels = pickle.load(file)
filename = './shuffled_balanced_cancer_labels.pickle'
with open(filename, 'rb') as file:
    shuffled_balanced_cancer_labels = pickle.load(file)

In [4]:
# Choose your phase
run_partial_inclusion = True 

if run_partial_inclusion:
    from data_partial_inclusion import get_partial_inclusion_data
    (train_x, train_y), (val_x, val_y), (eval_x, eval_y) = get_partial_inclusion_data(
        shuffled_reshaped_balanced_images, shuffled_balanced_cancer_labels, shuffled_balanced_center_labels, target_class=[0, 1]
    )
else:
    from data_full_exclusion import get_full_exclusion_data
    (train_x, train_y), (val_x, val_y), (eval_x, eval_y) = get_full_exclusion_data(
        shuffled_reshaped_balanced_images, shuffled_balanced_cancer_labels, shuffled_balanced_center_labels
    )

In [14]:
# ==============================================================================
# SECTION: Reproducibility & Experimental Seeds
# ==============================================================================
# To ensure the validity of our results across all reported architectures, 
# we enforce strict deterministic operations.

def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

# We iterate through established seeds (e.g., 42, 123, 7, 1024, 99) 
# as reported in the supplementary tables of our manuscript.
current_seed = 99
set_reproducibility(current_seed)

In [17]:
# Example: Running the  experiment for DenseNet
model = get_densenet_model()

optimizer = Adam(learning_rate=0.0001) 
model.compile(
    optimizer=optimizer, 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

print("Model Initialized for Deceptive Pipeline Analysis.")

Model Initialized for Deceptive Pipeline Analysis.


In [18]:
## model training
history = model.fit(
    train_x, 
    train_y,
    steps_per_epoch=len(train_y) // 16, 
    epochs=5, 
    shuffle=True,
    validation_split=0.1
)

Epoch 1/5
843/843 [==============================] - 54s 57ms/step - loss: 3.3909 - accuracy: 0.6580 - val_loss: 2.3881 - val_accuracy: 0.7704
Epoch 2/5
843/843 [==============================] - 47s 55ms/step - loss: 2.0385 - accuracy: 0.7304 - val_loss: 1.6813 - val_accuracy: 0.7748
Epoch 3/5
843/843 [==============================] - 46s 55ms/step - loss: 1.4270 - accuracy: 0.7858 - val_loss: 1.2010 - val_accuracy: 0.8096
Epoch 4/5
843/843 [==============================] - 46s 54ms/step - loss: 1.0191 - accuracy: 0.8229 - val_loss: 0.9563 - val_accuracy: 0.7978
Epoch 5/5
843/843 [==============================] - 38s 45ms/step - loss: 0.7667 - accuracy: 0.8539 - val_loss: 0.7721 - val_accuracy: 0.8193


In [ ]:
# ==============================================================================
# SECTION: Error Rate Calculation (External Evaluation)
# ==============================================================================
# We evaluate the model on external centers to observe the behavioral shift
# caused by site-specific shortcuts.

# --- Step 1: Generate Predictions ---
# We use the external evaluation subset defined in the partitioning phase.
predicted_probabilities = model.predict(eval_x)
predicted_labels = np.argmax(predicted_probabilities, axis=1)

# --- Step 2: Convert One-Hot Labels ---
true_labels = np.argmax(eval_y, axis=1)

# --- Step 3: Compute Confusion Matrix ---
cm = confusion_matrix(true_labels, predicted_labels)

# --- Step 4: Calculate Class-Specific Error Rates ---
# Class 0: LUAD (Lung Adenocarcinoma)
# Class 1: LUSC (Lung Squamous Cell Carcinoma)

# Error rate for C0: (False Positives for C1 / Total C0 samples)
error_rate_c0 = cm[0][1] / (cm[0][0] + cm[0][1])

# Error rate for C1: (False Positives for C0 / Total C1 samples)
error_rate_c1 = cm[1][0] / (cm[1][0] + cm[1][1])

print(f"Error Rate for LUAD (C0): {error_rate_c0:.4f}")
print(f"Error Rate for LUSC (C1): {error_rate_c1:.4f}")